# Spike 05 — Retrieval Quality Check

**Goal:** Ask real questions against the indexed documents and evaluate whether the answers are correct.

**Time box:** 1-2 hours

**Question to answer:** Does the end-to-end pipeline (OCR → PageIndex → LLM) return accurate, well-sourced answers?

**Done when:** You have a scored evaluation table showing which questions the system answered correctly.

---

### What we're testing

This spike closes the loop: we go from raw question → retrieved context → LLM answer → human judgment.  
The output is not production code — it's a **decision**: is the pipeline good enough to build on?

We use Groq (Llama 3.3 70B) as the answer-generation LLM — fast, free, and we already have it set up.

## Step 1 — Setup

In [1]:
import json
import os
from openai import OpenAI
from dotenv import load_dotenv
from pathlib import Path

load_dotenv(dotenv_path="../.env")

DATA_DIR = Path("../data")

# Load the PageIndex structure cache produced by spike_04
STRUCTURE_CACHE = DATA_DIR / "pageindex_structures.json"

if not STRUCTURE_CACHE.exists():
    raise FileNotFoundError(
        f"Structure cache not found at {STRUCTURE_CACHE}. "
        "Run spike_04_pageindex.ipynb first (Kernel → Restart & Run All)."
    )

structures = json.loads(STRUCTURE_CACHE.read_text(encoding="utf-8"))
en_nodes = structures.get("english", [])
ar_nodes = structures.get("arabic",  [])

print(f"Loaded structure cache:")
print(f"   English nodes : {len(en_nodes)}")
print(f"   Arabic nodes  : {len(ar_nodes)}")

# Groq — answer generation
groq_client = OpenAI(
    api_key  = os.getenv("GROQ_API_KEY"),
    base_url = "https://api.groq.com/openai/v1"
)
MODEL = "llama-3.3-70b-versatile"

print(f"Groq client ready — model: {MODEL}")

Loaded structure cache:
   English nodes : 13
   Arabic nodes  : 16
Groq client ready — model: llama-3.3-70b-versatile


## Step 2 — Retrieval Function

Send a question to PageIndex and get back the most relevant passages.

In [2]:
def search_structure(query: str, nodes: list, top_k: int = 5) -> list:
    """
    Score each PageIndex node by keyword overlap with the query.
    Returns top_k nodes as passage dicts with 'content', 'source', 'title' keys.
    """
    query_words = set(query.lower().split())
    scored = []

    for node in nodes:
        combined = " ".join([
            node.get("title",   ""),
            node.get("summary", ""),
            node.get("text",    ""),
        ]).lower()

        score = sum(1 for word in query_words if word in combined)
        if score > 0:
            scored.append((score, node))

    scored.sort(key=lambda x: x[0], reverse=True)
    top_nodes = [node for _, node in scored[:top_k]]

    # Normalise to passage dicts so generate_answer() doesn't need to know about node schema
    passages = []
    for node in top_nodes:
        passages.append({
            "content": node.get("text", ""),
            "title"  : node.get("title", ""),
            "source" : f"node_{node.get('node_id', '?')}",
        })
    return passages


def retrieve(question: str, language: str = "en", top_k: int = 5) -> list:
    """Select the right language structure and run keyword search."""
    nodes = en_nodes if language == "en" else ar_nodes
    results = search_structure(question, nodes, top_k)
    return results


# Quick sanity check
test = retrieve("livability indicators Madinah", language="en", top_k=2)
print(f"Sanity check — retrieved {len(test)} passages")
for p in test:
    print(f"  [{p['source']}] {p['title']} — {len(p['content'])} chars")

Sanity check — retrieved 2 passages
  [node_0002] Executive Summary — 7194 chars
  [node_0034] Preliminary Analysis of the Al Madinah Livable City Index — 2846 chars


## Step 3 — Answer Generation Function

Take retrieved passages + question → send to Groq → get a grounded answer with citations.

In [3]:
def generate_answer(question: str, passages: list, language: str = "en") -> str:
    """
    Generate a grounded answer from retrieved passages.
    The LLM is instructed to only use what's in the context — no hallucination.
    """
    if not passages:
        return "No relevant passages found to answer this question."

    # Build context block from retrieved passages
    context_parts = []
    for i, p in enumerate(passages, 1):
        # Try common key names for content
        content = p.get("content") or p.get("text") or p.get("passage") or str(p)
        source  = p.get("source") or p.get("doc_id") or p.get("id") or f"passage_{i}"
        context_parts.append(f"[Source {i}: {source}]\n{content}")

    context = "\n\n".join(context_parts)

    # Language-aware system prompt
    if language == "ar":
        system = (
            "أنت مساعد متخصص في تحليل تقارير التنمية الحضرية لمدينة المدينة المنورة. "
            "أجب على الأسئلة بالعربية فقط باستخدام المعلومات المقدمة. "
            "اذكر المصدر لكل معلومة. إذا لم تجد الإجابة في النصوص المقدمة، قل ذلك صراحة."
        )
    else:
        system = (
            "You are an expert analyst of Madinah urban development reports. "
            "Answer questions using ONLY the provided context passages. "
            "Cite the source number for each fact (e.g. [Source 1]). "
            "If the answer is not in the context, say so explicitly — do NOT guess."
        )

    messages = [
        {"role": "system", "content": system},
        {"role": "user",   "content": f"Context passages:\n\n{context}\n\nQuestion: {question}"}
    ]

    response = groq_client.chat.completions.create(
        model      = MODEL,
        messages   = messages,
        max_tokens = 1024
    )

    return response.choices[0].message.content


print("✅ generate_answer() defined")

✅ generate_answer() defined


## Step 4 — Full Pipeline Function

In [4]:
def ask(question: str, language: str = "en", verbose: bool = True) -> dict:
    """
    Full pipeline: question → retrieve → generate → return answer with sources.
    """
    if verbose:
        print(f"Q: {question}")
        print("-" * 60)

    # Step 1: Retrieve relevant passages
    passages = retrieve(question, language=language)
    if verbose:
        print(f"Retrieved {len(passages)} passages from PageIndex")

    # Step 2: Generate answer
    answer = generate_answer(question, passages, language=language)

    if verbose:
        print(f"\nAnswer:\n{answer}")
        print()

    return {
        "question": question,
        "answer"  : answer,
        "sources" : [p.get("source", "") for p in passages]
    }


print("✅ ask() pipeline ready")

✅ ask() pipeline ready


## Step 5 — Run the Evaluation Test Set

These are real questions a policy analyst or urban planner might ask about the Madinah report.  
After each answer, we score it manually: ✅ correct / ⚠️ partial / ❌ wrong.

In [5]:
# Evaluation questions — (question, language)
TEST_QUESTIONS = [
    # English
    ("What are the livability indicators used in this report?",            "en"),
    ("What is the population of Madinah according to the report?",         "en"),
    ("What sustainability goals were achieved in 2024?",                   "en"),
    ("Which neighborhoods scored highest on livability?",                  "en"),
    ("What are the key challenges facing Madinah's urban development?",    "en"),

    # Arabic
    ("ما هي مؤشرات قابلية العيش المستخدمة في هذا التقرير؟",              "ar"),
    ("ما هو عدد سكان المدينة المنورة وفقاً للتقرير؟",                     "ar"),
    ("ما هي أبرز إنجازات التنمية المستدامة في عام 2024؟",                 "ar"),
]

results = []

for i, (question, lang) in enumerate(TEST_QUESTIONS):
    print(f"\n{'='*65}")
    print(f"Test {i+1}/{len(TEST_QUESTIONS)} [{lang.upper()}]")
    result = ask(question, language=lang, verbose=True)
    results.append(result)


Test 1/8 [EN]
Q: What are the livability indicators used in this report?
------------------------------------------------------------
Retrieved 5 passages from PageIndex

Answer:
The livability indicators used in this report include [Source 1]:

1. Spatial and Historical Perspective Sector
2. Humanized Livable City Sector
3. Urban Mobility Sector
4. Affordable Decent Housing sector
5. Sustainable Environment Sector
6. Health Care Quality Sector
7. Quality Education Sector
8. Social Capital, Social Cohesion, and Inclusion Sector
9. A thriving and vibrant economic sector
10. Tourism and Entertainment Sector

Additionally, [Source 2] mentions the Al Madinah Livable City Index, which includes scores for each of these sectors. 

These indicators are also reflected in the table of contents [Source 4], which lists chapters corresponding to each sector.


Test 2/8 [EN]
Q: What is the population of Madinah according to the report?
------------------------------------------------------------
Re

## Step 6 — Score the Results

Run this cell after reading each answer. Score manually — you know the content of the reports.

In [8]:
# Fill in your scores after reading each answer above
# Options: 'correct', 'partial', 'wrong', 'unsure'
SCORES = [
    'correct',  # Q1 EN
    'correct',  # Q2 EN
    'correct',  # Q3 EN
    'correct',  # Q4 EN
    'correct',  # Q5 EN
    'partial',  # Q1 AR
    'partial',  # Q2 AR
    'partial',  # Q3 AR
]

# Summary
from collections import Counter
score_counts = Counter(SCORES)

total    = len(SCORES)
correct  = score_counts['correct']
partial  = score_counts['partial']
wrong    = score_counts['wrong']
unsure   = score_counts['unsure']

print("=" * 50)
print("SPIKE 05 — EVALUATION RESULTS")
print("=" * 50)
print(f"Total questions : {total}")
print(f"✅ Correct      : {correct}  ({correct/total*100:.0f}%)")
print(f"⚠️  Partial      : {partial}  ({partial/total*100:.0f}%)")
print(f"❌ Wrong        : {wrong}   ({wrong/total*100:.0f}%)")
print(f"❓ Unsure       : {unsure}   (score these manually)")
print()

effective_score = (correct + 0.5 * partial) / (total - unsure) * 100 if (total - unsure) > 0 else 0
print(f"Effective score : {effective_score:.0f}%")
print()

if effective_score >= 70:
    print("✅ SPIKE PASSED — Pipeline is good enough to build on")
    print("   Next: Apply Martin Fowler patterns (query rewriting, re-ranking, guards)")
elif effective_score >= 40:
    print("⚠️  SPIKE PARTIAL — Investigate failures before continuing")
    print("   Common fixes: improve OCR prompt, re-index with better chunking")
else:
    print("❌ SPIKE FAILED — Pipeline needs significant work")
    print("   Check: Did Spike 02/03 produce clean text? Is PageIndex indexing correctly?")

SPIKE 05 — EVALUATION RESULTS
Total questions : 8
✅ Correct      : 5  (62%)
⚠️  Partial      : 3  (38%)
❌ Wrong        : 0   (0%)
❓ Unsure       : 0   (score these manually)

Effective score : 81%

✅ SPIKE PASSED — Pipeline is good enough to build on
   Next: Apply Martin Fowler patterns (query rewriting, re-ranking, guards)


## Step 7 — Diagnose Failures

If any answers were wrong, investigate here.

In [7]:
# Pick any question index (0-based) to debug a bad answer
DEBUG_IDX = 0

if DEBUG_IDX < len(results):
    q, lang = TEST_QUESTIONS[DEBUG_IDX]

    print(f"=== DEBUGGING Q{DEBUG_IDX+1} ===")
    print(f"Question : {q}")
    print(f"Language : {lang}")
    print()

    passages = retrieve(q, language=lang, top_k=5)
    print(f"Raw passages retrieved: {len(passages)}")

    for i, p in enumerate(passages, 1):
        print(f"\n--- Passage {i} [{p['source']}] ---")
        print(f"Title   : {p['title']}")
        print(f"Content : {p['content'][:500]}")

    print("\n=== DIAGNOSIS ===")
    if not passages:
        print("No passages returned — query words may not appear in any node.")
        print("Try broadening the question or check that spike_04 ran on correct OCR output.")
    elif all(len(p.get("content", "")) < 50 for p in passages):
        print("Passages are very short — OCR quality issue? Check ocr_output/ files.")
    else:
        print("Passages look OK. Problem may be in answer generation.")
        print("Try adjusting the system prompt in generate_answer().")

=== DEBUGGING Q1 ===
Question : What are the livability indicators used in this report?
Language : en

Raw passages retrieved: 5

--- Passage 1 [node_0002] ---
Title   : Executive Summary
Content : # Executive Summary

This report comprehensively analyzes the livability of Al Madinah City and identifies strengths and areas for improvement through effective urban policies. Enhancing the city's livability can make it a role model in sustainability, resilience, inclusiveness, and innovation, achieving an ideal balance between society, environment, and economy.

**Spatial and Historical Perspective Sector:** Madinah is a global beacon of Islam, culture, and history. Therefore, the culture and 

--- Passage 2 [node_0034] ---
Title   : Preliminary Analysis of the Al Madinah Livable City Index
Content : # Preliminary Analysis of the Al Madinah Livable City Index

For the **Spatial and Historical Perspective** sector, Al Madinah City scores relatively well (81.0%) as the Global Beacon for Isla